In [3]:
! pip install -q langchain-anthropic tavily-python langchain_community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 409.5/409.5 kB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 62.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.5/49.5 kB 4.6 MB/s eta 0:00:00


In [4]:
import os

os.environ["ANTHROPIC_API_KEY"] = "YOUR_ANTHROPIC_API_KEY"
os.environ["TAVILY_API_KEY"] = "YOUR_TAVILY_API_KEY"

In [30]:
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.tools.tavily_search import TavilySearchResults

In [35]:
weeb_search = TavilySearchResults(k=3)

system = """
use os resultados encontrados pela ferramento de consulta para responder
de forma concisa a pergunta {pergunta} do usuario.

estruture sua resposta da seguinte forma:

1. Resposta para pergunta do usuario com base nos resultados encontrados : {seachs}
2. urls onde foi encontrado a resposta : {urls}
"""

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human",
         "Aqui esta a pergunta do usuario {pergunta}. \
         \n\n\
         Aqui estão os resultados da consulta: {seachs} \
         \n\n\
         Aqui estão as urls {urls}")
    ]
)

model = "claude-3-5-sonnet-20241022"
llm = ChatAnthropic(model=model,
                    temperature=0.1,
                    max_tokens=512)

chain = prompt | llm | StrOutputParser()

In [41]:
def response(query):
  searchs = weeb_search.invoke({"query": query})

  content = "\n".join([s["content"] for s in searchs])
  urls = [s["url"] for s in searchs]

  response = chain.invoke({"pergunta": query,
                           "seachs": content,
                           "urls": urls})
  return response

In [42]:
query = "qual valor de mercado da NVIDIA?"
res = response(query)
res

'Com base nos resultados encontrados, o valor de mercado atual da NVIDIA é de US$ 3,335 trilhões, tornando-a uma das empresas mais valiosas do mundo, tendo inclusive ultrapassado a Microsoft em valor de mercado.\n\nURLs relevantes:\n- https://www.infomoney.com.br/mercados/nvidia-ultrapassa-microsoft-em-valor-de-mercado-e-se-torna-a-mais-valiosa-do-mundo/\n- https://pt.investing.com/equities/nvidia-corp'

In [40]:
query = "qual nome do primeiro homem a pisar na lua"
res = response(query)
res

"1. Resposta para pergunta do usuário com base nos resultados encontrados:\nNeil Armstrong foi o primeiro homem a pisar na Lua, em 20 de julho de 1969, durante a missão Apollo 11. Ele era piloto militar, engenheiro aeroespacial e comandante da missão quando deu este histórico passo para a humanidade.\n\n2. URLs onde foi encontrada a resposta:\n['https://brasilescola.uol.com.br/fisica/primeiro-homem-pisar-na-lua.htm', 'https://www.cnnbrasil.com.br/tecnologia/quem-foi-neil-armstrong-primeiro-humano-a-pisar-na-lua/', 'https://www.todamateria.com.br/neil-armstrong/', 'https://pt.wikipedia.org/wiki/Neil_Armstrong', 'https://mundoeducacao.uol.com.br/historiageral/neil-armstrong.htm']"